# Code Intelligence & Generation

Companion notebook for the [Code Intelligence lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/12-code-intelligence-and-generation).

**The idea in one sentence.** LLM coding tools need code-aware plumbing: **AST-based
chunking** (split by function/class, not arbitrary lines), **BM25 search** over the
codebase (lexical retrieval that beats embeddings for exact symbol names), and a
**generate → validate → filter** loop that runs the generated code and keeps only what
passes.

The building blocks:

- **AST chunking:** parse the code and split on syntactic boundaries so a chunk is a
  whole function, not a fragment.
- **BM25 retrieval:** rank code by term overlap with the query.
- **Generate-validate-filter:** execute candidate tests/code and keep the ones that pass.

We build all three, **validate the AST chunker and BM25 ranking**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import re, ast
from collections import Counter
import numpy as np
rng = np.random.default_rng(0)

## 1 — AST-based code chunking

Splitting code by character count breaks function boundaries. We use Python's AST to extract function-level chunks.

In [ ]:
sample_module = '''
def add(a: int, b: int) -> int:
    """Return the sum of a and b."""
    return a + b

def multiply(x: float, y: float) -> float:
    """Return the product of x and y."""
    if y == 0:
        return 0.0
    return x * y

class Calculator:
    """Simple calculator class."""
    def __init__(self, initial=0.0):
        self.value = initial

    def reset(self):
        self.value = 0.0
        return self
'''

def ast_chunk_functions(source_code):
    """Extract all function and class definitions as separate chunks."""
    tree = ast.parse(source_code)
    lines = source_code.splitlines(keepends=True)
    chunks = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            start = node.lineno - 1
            end = node.end_lineno
            chunk_text = ''.join(lines[start:end])
            chunks.append({'name': node.name, 'type': type(node).__name__, 'code': chunk_text})
    return chunks

chunks = ast_chunk_functions(sample_module)
for c in chunks:
    print(f"[{c['type']}] {c['name']}")
    print(c['code'][:80].replace('\n', ' ') + '...')
    print()

### Validate: AST chunking splits on function/class boundaries

Line-based chunking can cut a function in half; AST chunking parses the code and returns
whole syntactic units. We confirm the chunker finds the two top-level functions and the
class in the sample module (not fragments).

In [ ]:
import ast as _ast, textwrap
chunks = ast_chunk_functions(sample_module)
names = [ch['name'] for ch in chunks]
print(f'{len(chunks)} chunks; defs found: {names}')
assert 'add' in names and 'multiply' in names and 'Calculator' in names, 'chunks preserve whole defs'
# every chunk is a whole, parseable def (dedent nested methods first) -- not a mid-line fragment
for ch in chunks:
    _ast.parse(textwrap.dedent(ch['code']))
print('\n✅ AST chunking returns whole functions/classes, each independently parseable')

## 2 — BM25 code search (without external libraries)

We implement a simple BM25 retrieval over function docstrings + names as a proxy for code semantic search.

In [ ]:
def tokenize(text):
    return re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*', text.lower())

def bm25(query, corpus, k1=1.5, b=0.75):
    """BM25 ranking of corpus documents against a query."""
    tokens = [tokenize(doc) for doc in corpus]
    avg_len = np.mean([len(t) for t in tokens])
    query_tokens = set(tokenize(query))
    n = len(corpus)
    scores = np.zeros(n)
    for term in query_tokens:
        df = sum(1 for t in tokens if term in t)
        if df == 0: continue
        idf = np.log((n - df + 0.5) / (df + 0.5) + 1)
        for i, t in enumerate(tokens):
            tf = t.count(term)
            tf_norm = tf * (k1+1) / (tf + k1*(1 - b + b*len(t)/avg_len))
            scores[i] += idf * tf_norm
    return scores

# Build a tiny code corpus from our chunks
corpus = [f"{c['name']}: {c['code']}" for c in chunks]
query = "multiply two numbers together"
scores = bm25(query, corpus)
ranked = np.argsort(-scores)
print(f"Query: '{query}'")
print("Ranked results:")
for rank, i in enumerate(ranked):
    print(f"  {rank+1}. [{scores[i]:.3f}] {chunks[i]['name']}")

### Validate: BM25 ranks the relevant code highest

BM25 scores documents by query-term overlap (with length normalisation). For a query about
multiplication, the `multiply` function should rank above unrelated code — the lexical
retrieval that finds exact symbol names embeddings often miss.

In [ ]:
code_corpus = [
    'def multiply(x, y): return x * y',
    'def add(a, b): return a + b',
    'class Logger: def info(self, msg): print(msg)',
    'def divide(p, q): return p / q',
]
scores = bm25('multiply two numbers product', code_corpus)
ranked = int(np.argmax(scores))
print('BM25 scores:', [round(float(s), 3) for s in scores])
print('top-ranked snippet:', code_corpus[ranked])
assert ranked == 0, 'the multiply snippet should rank highest for a multiply query'
print('\n✅ BM25 surfaces the code whose tokens overlap the query — great for exact symbols')

## 3 — Generate → validate → filter for test generation

The Meta TestGen-LLM pattern: generate many test candidates, execute them, keep only passing ones.

In [ ]:
# We simulate the generate-validate-filter loop with synthetic test candidates

def target_function(x):
    """The function we want to test: clamp x to [0, 100]."""
    return max(0, min(100, x))

# Simulate LLM-generated test candidates (some will "fail" = wrong expected value)
test_candidates = [
    ("test_zero",    lambda: assert_eq(target_function(0), 0)),
    ("test_fifty",   lambda: assert_eq(target_function(50), 50)),
    ("test_above",   lambda: assert_eq(target_function(150), 100)),
    ("test_below",   lambda: assert_eq(target_function(-10), 0)),
    ("test_wrong",   lambda: assert_eq(target_function(50), 51)),   # WRONG expected value
    ("test_edge",    lambda: assert_eq(target_function(100), 100)),
    ("test_float",   lambda: assert_eq(target_function(99.9), 99.9)),
]

def assert_eq(a, b): assert a == b, f"{a} != {b}"

def run_and_filter_tests(candidates):
    passing, failing = [], []
    for name, test_fn in candidates:
        try:
            test_fn()
            passing.append(name)
        except AssertionError as e:
            failing.append((name, str(e)))
    return passing, failing

passing, failing = run_and_filter_tests(test_candidates)
print(f"Generated: {len(test_candidates)} tests")
print(f"Passing:   {len(passing)} tests → {passing}")
print(f"Failing:   {len(failing)} tests:")
for name, err in failing:
    print(f"  {name}: {err}")
print(f"\nFilter rate: {len(passing)/len(test_candidates):.0%} kept")

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **line-based chunking** | splits functions mid-body; the model sees fragments |
| **embeddings-only search** | blurs exact identifiers; combine with BM25 (demo) |
| **trusting generated code** | always validate by execution; the model hallucinates APIs |
| **executing untrusted code** | sandbox generated code — it may be wrong or malicious |
| **stale index** | re-index on code changes or retrieval returns deleted symbols |

Demo: BM25 exactly matches an identifier name where embeddings would blur it.

In [ ]:
# Why BM25 (lexical) still matters vs embeddings: exact identifier names. A query for a
# specific function name must match it EXACTLY — embeddings blur symbols, BM25 nails them.
corpus2 = ['def parse_iso_timestamp(s): ...', 'def format_date(d): ...', 'def tokenize(t): ...']
s = bm25('parse_iso_timestamp', corpus2)
print('query "parse_iso_timestamp" BM25 scores:', [round(float(x), 3) for x in s])
assert int(np.argmax(s)) == 0, 'BM25 exactly matches the identifier'
print('\nFor exact symbol/identifier lookup, lexical BM25 beats fuzzy embeddings -> hybrid search wins.')

## ✏️ Your turn

**Exercise.** Implement `jaccard_similarity(tokens_a, tokens_b)` — the Jaccard coefficient between two sets of tokens. This is the fast, no-ML method Copilot uses for initial context retrieval: find files whose tokens maximally overlap with the current file.

In [ ]:
def jaccard_similarity(tokens_a, tokens_b):
    """
    tokens_a, tokens_b: iterables of string tokens
    Returns: |intersection| / |union|
    """
    # TODO(you): compute Jaccard similarity between the two token sets
    return ...

# Compare two code files by their token overlap
file1 = tokenize("def multiply(x, y): return x * y")
file2 = tokenize("def add(a, b): return a + b")
file3 = tokenize("result = multiply(3, 4) + multiply(2, 2)")

print(f"file1 vs file2 (different ops): {jaccard_similarity(file1, file2):.4f}")
print(f"file1 vs file3 (same 'multiply'): {jaccard_similarity(file1, file3):.4f}")
print("file3 should be more similar to file1 (shares 'multiply')")

In [ ]:
# Assertion
assert abs(jaccard_similarity(['a','b','c'], ['b','c','d']) - 2/4) < 1e-6
assert jaccard_similarity(['x'], ['y']) == 0.0
assert jaccard_similarity(['a'], ['a']) == 1.0
print("✓ jaccard_similarity correct")

<details><summary>Solution</summary>

```python
def jaccard_similarity(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union > 0 else 0.0
```
</details>

## Key takeaways

- **Chunk code by AST, not lines:** each chunk is a whole function/class, independently
  parseable (verified) — no fragments to confuse the model.
- **BM25 ranks by lexical overlap** and nails exact symbol names embeddings blur
  (verified + demo) — that's why code search is often *hybrid* (BM25 + embeddings).
- **Generate → validate → filter:** run the generated code/tests and keep only what
  passes — the model proposes, execution disposes.
- **Code intelligence is code-aware plumbing** around the model, not just prompting.